# Анализ и прогноз информационных событий

## Конфигурация

In [2]:
# ── Clustering ──────────────────────────────────────────────
MIN_CLUSTER_SIZE = 5
TIME_WEIGHT      = 0.3
UMAP_COMPONENTS  = 20
UMAP_NEIGHBORS   = 15

# ── Dynamics ─────────────────────────────────────────────────
BURST_GAP_DAYS         = 7     # минимальный перерыв между вспышками
MULTI_BURST_THRESHOLD  = 3     # n_bursts >= N → multi_burst
LONG_TAIL_RATIO        = 3     # decay > rise * N → long_tail
SLOW_BUILD_RATIO       = 3     # rise > decay * N → slow_build

# ── Forecast ─────────────────────────────────────────────────
MIN_ACTIVE_DAYS        = 30
TRAIN_RATIO            = 0.8
EXP_SMOOTHING_LEVEL    = 0.3
PROPHET_CHANGEPOINT    = 0.1
PROPHET_SEASONALITY    = 1.0

# ── Paths ────────────────────────────────────────────────────
PATH        = "posts.json"
ORIGIN_PATH = "hf://datasets/ScoutieAutoML/russian-news-telegram-dataset/scoutieRussianNewsTelegramDataset.json"


## Loader

In [3]:
import os
import pandas as pd

if not os.path.exists(PATH):
    raw_df = pd.read_json(ORIGIN_PATH, lines=True)
    raw_df.to_json(PATH)

print("Dataset file ready:", PATH)


Dataset file ready: posts.json


## Clustering

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import MinMaxScaler
from umap import UMAP


def load_data(path):
    df = pd.read_json(path, orient="columns")
    df = df[df["spam"] == "NOT SPAM"]
    df = df[df["language"] == "rus"]
    df = df.dropna(subset=["vector", "create_time", "text"])
    df = df.sort_values("create_time").reset_index(drop=True)
    df["datetime"] = pd.to_datetime(df["create_time"], unit="ms")
    return df


def parse_vectors(df):
    vectors = np.vstack(df["vector"].apply(lambda v: np.array(v, dtype=np.float32)).values)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    return vectors / norms


def normalize_timestamps(df):
    scaler = MinMaxScaler()
    ts = df["create_time"].values.astype(np.float64).reshape(-1, 1)
    return scaler.fit_transform(ts).flatten()


def reduce_dimensions(vectors, n_components=UMAP_COMPONENTS):
    reducer = UMAP(n_components=n_components, n_neighbors=UMAP_NEIGHBORS,
                   metric="cosine", random_state=42, low_memory=True, verbose=True)
    return reducer.fit_transform(vectors)


def build_feature_matrix(vectors_reduced, timestamps_norm, time_weight=TIME_WEIGHT):
    return np.hstack([vectors_reduced, (timestamps_norm * time_weight).reshape(-1, 1)])


def cluster_events(features, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=5):
    return HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples,
                   metric="euclidean").fit_predict(features)


def extract_ner_columns(df):
    df = df.copy()
    def get_by_label(ners, labels):
        if not isinstance(ners, list): return []
        return list({e["lemma"] for e in ners if e.get("label") in labels})
    df["ner_persons"]   = df["ners"].apply(lambda x: get_by_label(x, {"PER"}))
    df["ner_orgs"]      = df["ners"].apply(lambda x: get_by_label(x, {"ORG"}))
    df["ner_locations"] = df["ners"].apply(lambda x: get_by_label(x, {"LOC", "GPE"}))
    return df


def summarize_clusters(df):
    def top_entities(series, n=5):
        from collections import Counter
        all_ents = [e for lst in series for e in lst]
        return [e for e, _ in Counter(all_ents).most_common(n)]
    grouped = df[df["event_cluster"] != -1].groupby("event_cluster")
    summary = grouped.agg(
        post_count=("id","count"), date_start=("datetime","min"), date_end=("datetime","max"),
        avg_views=("views","mean"),
        emo_positive=("emo", lambda x: (x=="POSITIVE").sum()),
        emo_negative=("emo", lambda x: (x=="NEGATIVE").sum()),
        emo_neutral =("emo", lambda x: (x=="NEUTRAL").sum()),
    ).reset_index()
    summary["duration_days"] = (summary["date_end"] - summary["date_start"]).dt.days
    ner_agg = grouped.apply(lambda g: pd.Series({
        "top_persons":   top_entities(g["ner_persons"]),
        "top_orgs":      top_entities(g["ner_orgs"]),
        "top_locations": top_entities(g["ner_locations"]),
    })).reset_index()
    return summary.merge(ner_agg, on="event_cluster").sort_values("post_count", ascending=False).reset_index(drop=True)


def run_pipeline(path):
    print("Loading data...")
    df = load_data(path)
    print(f"Posts after filtering: {len(df)}")
    print("Parsing vectors...")
    vectors = parse_vectors(df)
    print(f"Reducing dimensions with UMAP (768 -> {UMAP_COMPONENTS})...")
    vectors_reduced = reduce_dimensions(vectors)
    print(f"Reduced shape: {vectors_reduced.shape}")
    print("Building feature matrix with time...")
    timestamps_norm = normalize_timestamps(df)
    features = build_feature_matrix(vectors_reduced, timestamps_norm)
    print("Clustering with HDBSCAN...")
    labels = cluster_events(features)
    df["event_cluster"] = labels
    n_events = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise  = (labels == -1).sum()
    print(f"Events found:  {n_events}")
    print(f"Noise posts:   {n_noise} ({n_noise / len(df) * 100:.1f}%)")
    print("Extracting NER...")
    df = extract_ner_columns(df)
    print("Building summary...")
    summary = summarize_clusters(df)
    return df, summary


posts, events = run_pipeline(PATH)
print("\nTop 10 events:")
cols = ["event_cluster","post_count","date_start","date_end","duration_days","top_persons","top_orgs"]
print(events[cols].head(10).to_string())


Loading data...
Posts after filtering: 92939
Parsing vectors...
Reducing dimensions with UMAP (768 -> 20)...
UMAP(angular_rp_forest=True, metric='cosine', n_components=20, n_jobs=1, random_state=42, verbose=True)
Tue May  5 15:01:44 2026 Construct fuzzy simplicial set
Tue May  5 15:01:44 2026 Finding Nearest Neighbors


/Users/gwkeo/projects/diploma/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Tue May  5 15:01:44 2026 Building RP forest with 20 trees
Tue May  5 15:01:51 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	 3  /  17
	 4  /  17
	 5  /  17
	 6  /  17
	 7  /  17
	Stopping threshold met -- exiting after 7 iterations
Tue May  5 15:02:10 2026 Finished Nearest Neighbor Search


## Dynamics

In [ ]:
import numpy as np
import pandas as pd


def build_daily_intensity(df):
    total_per_day = df.groupby(df["datetime"].dt.date)["id"].count().rename("total_posts")
    total_per_day.index = pd.to_datetime(total_per_day.index)
    events = df[df["event_cluster"] != -1].copy()
    events["date"] = events["datetime"].dt.floor("D")
    daily = (
        events.groupby(["event_cluster","date"])
        .agg(post_count=("id","count"), avg_views=("views","mean"),
             emo_positive=("emo", lambda x: (x=="POSITIVE").sum()),
             emo_negative=("emo", lambda x: (x=="NEGATIVE").sum()),
             emo_neutral =("emo", lambda x: (x=="NEUTRAL").sum()))
        .reset_index()
    )
    daily = daily.merge(total_per_day.rename("total_posts"), left_on="date", right_index=True)
    daily["relative_intensity"] = daily["post_count"] / daily["total_posts"]
    return daily


def compute_event_features(daily):
    features = []
    for cluster_id, group in daily.groupby("event_cluster"):
        group = group.sort_values("date")
        peak_idx   = group["post_count"].idxmax()
        peak_date  = group.loc[peak_idx, "date"]
        peak_value = group.loc[peak_idx, "post_count"]
        rise_days  = (peak_date - group[group["date"] <= peak_date]["date"].min()).days
        decay_days = (group[group["date"] >= peak_date]["date"].max() - peak_date).days
        gaps       = group["date"].diff().dt.days.dropna()
        n_bursts   = int((gaps > BURST_GAP_DAYS).sum()) + 1
        features.append({
            "event_cluster": cluster_id,
            "total_days":    (group["date"].max() - group["date"].min()).days,
            "active_days":   len(group),
            "peak_date":     peak_date,
            "peak_posts":    peak_value,
            "peak_relative": group.loc[peak_idx, "relative_intensity"],
            "rise_days":     rise_days,
            "decay_days":    decay_days,
            "max_gap_days":  gaps.max(),
            "n_bursts":      n_bursts,
            "total_posts":   group["post_count"].sum(),
            "total_views":   (group["avg_views"] * group["post_count"]).sum(),
        })
    return pd.DataFrame(features).sort_values("total_posts", ascending=False).reset_index(drop=True)


def classify_pattern(row):
    if row["n_bursts"] >= MULTI_BURST_THRESHOLD:
        return "multi_burst"
    if row["n_bursts"] == 2:
        return "two_burst"
    if row["decay_days"] > row["rise_days"] * LONG_TAIL_RATIO:
        return "long_tail"
    if row["rise_days"] > row["decay_days"] * SLOW_BUILD_RATIO:
        return "slow_build"
    return "single_peak"


print("Building daily intensity...")
daily = build_daily_intensity(posts)

print("Computing event features...")
event_features = compute_event_features(daily)
event_features["pattern"] = event_features.apply(classify_pattern, axis=1)

print("\nPattern distribution:")
print(event_features["pattern"].value_counts().to_string())

print("\nTop 10 events:")
cols = ["event_cluster","total_posts","active_days","peak_date","peak_posts","n_bursts","pattern"]
print(event_features[cols].head(10).to_string())


Building daily intensity...
Computing event features...

Pattern distribution:
pattern
multi_burst    1784
two_burst       312
long_tail       128
single_peak      94
slow_build       37

Top 10 events:
   event_cluster  total_posts  active_days  peak_date  peak_posts  n_bursts      pattern
0           2073         1024          261 2024-05-12         156        18  multi_burst
1           1597          375          167 2024-08-08          13        21  multi_burst
2           1112          318          171 2024-07-28           7        18  multi_burst
3           1821          306          196 2024-10-22           5        14  multi_burst
4            363          279           62 2023-10-07          55        14  multi_burst
5           1845          278          194 2023-06-10           4        25  multi_burst
6           1255          271          176 2024-10-05           6        39  multi_burst
7           2279          268          150 2024-10-20           8        21  multi_bu

## Именование кластеров

In [ ]:
import re
import pandas as pd
import nltk
from collections import Counter

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

_ru_stopwords = set(stopwords.words("russian"))

# ── Доменные стоп-слова (медиа-бренды, не покрытые nltk) ────
STOP_ORGS = {
    "mash", "shot", "baza", "сми", "upd", "впш", "рен", "риа",
    "тасс", "интерфакс", "readovka", "базаbazon",
}

EMO_LABEL = {
    "POSITIVE": "📈",
    "NEGATIVE": "📉",
    "NEUTRAL":  "📋",
}


def dominant_emo(row) -> str:
    pos = row.get("emo_positive", 0)
    neg = row.get("emo_negative", 0)
    neu = row.get("emo_neutral",  0)
    dominant = max([("POSITIVE", pos), ("NEGATIVE", neg), ("NEUTRAL", neu)],
                   key=lambda x: x[1])[0]
    return EMO_LABEL.get(dominant, "")


def extract_top_terms(series: pd.Series, stopset: set, n: int = 3) -> list[str]:
    # NER-данные уже лемматизированы, дополнительная морфология не нужна
    all_terms = [
        term.strip().title()
        for lst in series
        for term in lst
        if len(term.strip()) > 2
        and term.strip().lower() not in stopset
        and term.strip().lower() not in _ru_stopwords
    ]
    return [term for term, _ in Counter(all_terms).most_common(n)]


def extract_keywords_from_texts(texts: pd.Series, n: int = 3) -> list[str]:
    words = []
    for text in texts.dropna():
        tokens = re.findall(r"[а-яёА-ЯЁ]{4,}", str(text))
        for t in tokens:
            if t.lower() not in _ru_stopwords:
                words.append(t.lower())
    return [w.title() for w, _ in Counter(words).most_common(n)]


def generate_cluster_name(row: dict, posts_sample: pd.DataFrame) -> str:
    parts = []

    persons   = extract_top_terms(pd.Series([row.get("top_persons",   [])]), set(),      n=2)
    orgs      = extract_top_terms(pd.Series([row.get("top_orgs",      [])]), STOP_ORGS,  n=2)
    locations = extract_top_terms(pd.Series([row.get("top_locations", [])]), set(),      n=1)

    if persons:
        parts.extend(persons[:1])
    if locations:
        parts.extend(locations[:1])
    if orgs and len(parts) < 2:
        parts.extend(orgs[:1])

    if len(parts) < 2 and not posts_sample.empty:
        keywords = extract_keywords_from_texts(posts_sample["text"], n=3)
        for kw in keywords:
            if kw.lower() not in [p.lower() for p in parts]:
                parts.append(kw)
            if len(parts) >= 2:
                break

    if not parts:
        parts = [f"Событие #{row['event_cluster']}"]

    name = " / ".join(parts[:3])

    date_start = str(row.get("date_start", ""))[:7]
    date_end   = str(row.get("date_end",   ""))[:7]
    period     = f"{date_start} – {date_end}" if date_start and date_end else ""

    emo = dominant_emo(row)

    return f"{emo} {name} ({period})" if period else f"{emo} {name}"


def build_cluster_names(events: pd.DataFrame, posts: pd.DataFrame) -> pd.DataFrame:
    names = []
    for _, row in events.iterrows():
        cluster_id   = row["event_cluster"]
        posts_sample = posts[posts["event_cluster"] == cluster_id].head(50)
        name         = generate_cluster_name(row.to_dict(), posts_sample)
        names.append({"event_cluster": cluster_id, "cluster_name": name})

    names_df = pd.DataFrame(names)
    return events.merge(names_df, on="event_cluster")


def print_named_events(events_named: pd.DataFrame, top_n: int = 20) -> None:
    print(f"\nТоп-{top_n} событий с названиями:\n")
    available_cols = ["event_cluster", "cluster_name", "post_count", "date_start", "date_end", "pattern"]
    cols = [c for c in available_cols if c in events_named.columns]
    has_pattern = "pattern" in events_named.columns
    subset = events_named.nlargest(top_n, "post_count")[cols]
    for _, row in subset.iterrows():
        pattern_str = f" | {row['pattern']}" if has_pattern else ""
        print(f"  #{row['event_cluster']:5d} | {row['cluster_name']:<60s} | "
              f"{row['post_count']:5d} постов{pattern_str}")

In [ ]:
events_named = build_cluster_names(events, posts)
events_named = events_named.merge(
    event_features[["event_cluster", "pattern", "total_posts", "n_bursts"]],
    on="event_cluster",
    how="left"
)
print_named_events(events_named)

## Forecast

In [ ]:
import warnings
import numpy as np
import pandas as pd
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")


def prepare_prophet_df(group):
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    ts = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)
    return ts.reset_index().rename(columns={"index": "ds", "relative_intensity": "y"})


def train_test_split_ts(df, train_ratio=TRAIN_RATIO):
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx], df.iloc[split_idx:]


def fit_prophet(train):
    model = Prophet(
        yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
        changepoint_prior_scale=PROPHET_CHANGEPOINT,
        seasonality_prior_scale=PROPHET_SEASONALITY,
    )
    model.fit(train)
    return model


def predict_prophet(model, test):
    forecast = model.predict(test[["ds"]].copy())
    return np.clip(forecast["yhat"].values, 0, None), forecast


def predict_exp_smoothing(train, test):
    from statsmodels.tsa.holtwinters import SimpleExpSmoothing
    model = SimpleExpSmoothing(train["y"].values).fit(
        smoothing_level=EXP_SMOOTHING_LEVEL, optimized=False)
    return np.clip(model.forecast(len(test)), 0, None), None


def predict_arima(train, test, order=(5, 1, 0)):
    model = ARIMA(train["y"].values, order=order).fit()
    predicted = model.forecast(steps=len(test))
    return np.clip(predicted, 0, None), None


def compute_metrics(actual, predicted):
    mae  = mean_absolute_error(actual, predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5
    ab   = (actual > 0).astype(int)
    pb   = (predicted > actual.mean() * 0.3).astype(int)
    tp = ((ab==1)&(pb==1)).sum(); fp = ((ab==0)&(pb==1)).sum(); fn = ((ab==1)&(pb==0)).sum()
    precision = tp/(tp+fp) if (tp+fp)>0 else 0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {"mae": mae, "rmse": rmse, "precision": precision, "recall": recall, "f1": f1}


def _print_metrics(df):
    print(f"  MAE:       {df['mae'].mean():.6f}")
    print(f"  RMSE:      {df['rmse'].mean():.6f}")
    print(f"  Precision: {df['precision'].mean():.3f}")
    print(f"  Recall:    {df['recall'].mean():.3f}")
    print(f"  F1:        {df['f1'].mean():.3f}")


def run_forecast_with_model(daily, event_features, model_type_override=None):
    pattern_map = event_features.set_index("event_cluster")["pattern"].to_dict()
    eligible = [c for c in daily["event_cluster"].unique()
                if len(daily[daily["event_cluster"]==c]) >= MIN_ACTIVE_DAYS]
    print(f"Eligible events: {len(eligible)}")

    results, forecasts = [], []
    for i, cluster_id in enumerate(eligible):
        group      = daily[daily["event_cluster"]==cluster_id].sort_values("date")
        prophet_df = prepare_prophet_df(group)
        if len(prophet_df) < MIN_ACTIVE_DAYS: continue
        train, test = train_test_split_ts(prophet_df)
        if len(test) < 5: continue

        pattern    = pattern_map.get(cluster_id, "multi_burst")
        model_type = model_type_override or ("prophet" if pattern != "multi_burst" else "exp_smoothing")

        try:
            if model_type == "prophet":
                predicted, forecast_df = predict_prophet(fit_prophet(train), test)
            elif model_type == "arima":
                predicted, forecast_df = predict_arima(train, test)
            else:
                predicted, forecast_df = predict_exp_smoothing(train, test)

            actual  = test["y"].values
            metrics = compute_metrics(actual, predicted)
            metrics.update({"event_cluster": cluster_id, "pattern": pattern,
                            "model_used": model_type, "train_days": len(train), "test_days": len(test)})
            results.append(metrics)

            fc_df            = test[["ds"]].copy()
            fc_df["actual"]  = actual
            fc_df["yhat"]    = predicted
            if forecast_df is not None and "yhat_lower" in forecast_df.columns:
                fc_df["yhat_lower"] = np.clip(forecast_df["yhat_lower"].values, 0, None)
                fc_df["yhat_upper"] = np.clip(forecast_df["yhat_upper"].values, 0, None)
            else:
                fc_df["yhat_lower"] = predicted
                fc_df["yhat_upper"] = predicted
            fc_df["event_cluster"] = cluster_id
            forecasts.append(fc_df)

        except Exception as e:
            print(f"  Skipped {cluster_id}: {e}")

        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(eligible)} done...")

    return pd.concat(forecasts, ignore_index=True), pd.DataFrame(results)


# Прогон 1: выбор модели по паттерну (exp_smoothing для multi_burst, prophet для остальных)
print("=== Run 1: pattern-based model selection ===")
forecast_df, forecast_metrics = run_forecast_with_model(daily, event_features)
_print_metrics(forecast_metrics)

# Прогон 2: Prophet для всех событий
print("\n=== Run 2: Prophet for all events ===")
forecast_df_prophet, forecast_metrics_prophet = run_forecast_with_model(
    daily, event_features, model_type_override="prophet")
_print_metrics(forecast_metrics_prophet)

# Прогон 3: ARIMA(5,1,0) как baseline
print("\n=== Run 3: ARIMA(5,1,0) baseline ===")
forecast_df_arima, forecast_metrics_arima = run_forecast_with_model(
    daily, event_features, model_type_override="arima")
_print_metrics(forecast_metrics_arima)

print("\n=== Comparison by pattern ===")
for pattern in forecast_metrics["pattern"].unique():
    m1 = forecast_metrics[forecast_metrics["pattern"]==pattern]["f1"].mean()
    m2 = forecast_metrics_prophet[forecast_metrics_prophet["pattern"]==pattern]["f1"].mean()
    m3 = forecast_metrics_arima[forecast_metrics_arima["pattern"]==pattern]["f1"].mean()
    print(f"  {pattern:15s}  exp_smooth/pattern F1={m1:.3f}  prophet F1={m2:.3f}  arima F1={m3:.3f}")

## Visualization

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime


PATTERN_RU = {
    "multi_burst": "Многократные вспышки",
    "two_burst":   "Две вспышки",
    "long_tail":   "Длинный хвост",
    "slow_build":  "Медленный рост",
    "single_peak": "Одиночный пик",
}

MODEL_STYLE = {
    "Prophet":        {"color": "#E74C3C", "dash": "dash"},
    "Exp. Smoothing": {"color": "#F39C12", "dash": "dot"},
    "ARIMA":          {"color": "#9B59B6", "dash": "dashdot"},
}


def plot_event_forecast_multi(cluster_id, daily, event_features, name_map, model_forecasts):
    feat       = event_features[event_features["event_cluster"] == cluster_id].iloc[0].to_dict()
    event_name = name_map.get(cluster_id, f"Событие #{cluster_id}")

    any_fc     = next(iter(model_forecasts.values()))
    any_fc     = any_fc[any_fc["event_cluster"] == cluster_id].sort_values("ds")
    split_date = any_fc["ds"].min()

    group      = daily[daily["event_cluster"] == cluster_id].sort_values("date")
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    full_ts    = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)
    train_ts   = full_ts[full_ts.index < split_date]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=train_ts.index, y=train_ts.values, name="Train (факт)",
        line=dict(color="#4C9BE8", width=1.5), opacity=0.6,
    ))
    fig.add_trace(go.Scatter(
        x=any_fc["ds"], y=any_fc["actual"], name="Test (факт)",
        line=dict(color="#2ECC71", width=2.5),
    ))

    for model_label, fc_df in model_forecasts.items():
        fc = fc_df[fc_df["event_cluster"] == cluster_id].sort_values("ds")
        if fc.empty:
            continue
        style = MODEL_STYLE.get(model_label, {"color": "#888", "dash": "dash"})
        fig.add_trace(go.Scatter(
            x=fc["ds"], y=np.clip(fc["yhat"], 0, None),
            name=model_label,
            line=dict(color=style["color"], width=2, dash=style["dash"]),
        ))
        if "yhat_lower" in fc.columns and (fc["yhat_lower"] != fc["yhat"]).any():
            fig.add_trace(go.Scatter(
                x=pd.concat([fc["ds"], fc["ds"].iloc[::-1]]),
                y=pd.concat([
                    np.clip(fc["yhat_upper"], 0, None),
                    np.clip(fc["yhat_lower"], 0, None).iloc[::-1],
                ]),
                fill="toself",
                fillcolor=style["color"].replace("#", "rgba(") + ",0.08)",
                line=dict(color="rgba(0,0,0,0)"),
                showlegend=False,
            ))

    fig.add_shape(
        type="line", x0=str(split_date), x1=str(split_date), y0=0, y1=1, yref="paper",
        line=dict(dash="dot", color="gray", width=1.5),
    )
    fig.add_annotation(
        x=str(split_date), y=1, yref="paper", text="Train / Test",
        showarrow=False, xanchor="left", font=dict(color="gray", size=11),
    )
    fig.update_layout(
        title=(
            f"{event_name}<br>"
            f"<sup>Паттерн: {PATTERN_RU.get(feat['pattern'], feat['pattern'])} | "
            f"Активных дней: {feat['active_days']} | "
            f"Вспышек: {feat['n_bursts']} | "
            f"Пик: {str(feat['peak_date'])[:10]}</sup>"
        ),
        xaxis_title="Дата", yaxis_title="Относительная интенсивность",
        legend=dict(orientation="h", y=-0.2),
        height=480, template="plotly_white", hovermode="x unified",
    )
    return fig


def plot_model_comparison(metrics_dict):
    """metrics_dict: {label: metrics_df}"""
    colors = ["#4C9BE8", "#E74C3C", "#9B59B6", "#F39C12"]
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE", "RMSE", "F1"])
    for i, (label, metrics) in enumerate(metrics_dict.items()):
        agg = metrics.groupby("pattern")[["mae", "rmse", "f1"]].mean().reset_index()
        agg["pattern_ru"] = agg["pattern"].map(PATTERN_RU).fillna(agg["pattern"])
        color = colors[i % len(colors)]
        for col, row_i in [("mae", 1), ("rmse", 2), ("f1", 3)]:
            fig.add_trace(go.Bar(
                x=agg["pattern_ru"], y=agg[col], name=label,
                marker_color=color, opacity=0.85,
                showlegend=(row_i == 1),
            ), row=1, col=row_i)
    fig.update_layout(
        title="Сравнение моделей по паттернам",
        height=420, template="plotly_white", barmode="group",
        legend=dict(orientation="h", y=-0.2),
    )
    return fig


def plot_top_events_timeline(daily, event_features, name_map, top_n=8):
    top_clusters = event_features.nlargest(top_n, "total_posts")["event_cluster"].tolist()
    colors = px.colors.qualitative.Set1
    fig = go.Figure()
    for i, cluster_id in enumerate(top_clusters):
        group      = daily[daily["event_cluster"] == cluster_id].sort_values("date")
        full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
        ts         = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)
        label      = name_map.get(cluster_id, f"Событие #{cluster_id}")
        fig.add_trace(go.Scatter(
            x=ts.index, y=ts.values, name=label,
            line=dict(color=colors[i % len(colors)], width=1.5), opacity=0.85,
        ))
    fig.update_layout(
        title=f"Динамика топ-{top_n} событий",
        xaxis_title="Дата", yaxis_title="Относительная интенсивность",
        height=520, template="plotly_white", hovermode="x unified",
        legend=dict(orientation="h", y=-0.3),
    )
    return fig


def plot_metrics_distribution(forecast_metrics):
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE", "RMSE", "F1 Score"])
    for col, name, color, row_i in [
        ("mae",  "MAE",  "#4C9BE8", 1),
        ("rmse", "RMSE", "#E74C3C", 2),
        ("f1",   "F1",   "#2ECC71", 3),
    ]:
        fig.add_trace(go.Histogram(x=forecast_metrics[col], name=name,
                                   marker_color=color, opacity=0.8, nbinsx=30),
                      row=1, col=row_i)
    fig.update_layout(title="Распределение метрик (pattern-based)", height=350,
                      template="plotly_white", showlegend=False)
    return fig


def plot_pattern_distribution(event_features):
    counts = event_features["pattern"].value_counts().reset_index()
    counts.columns = ["pattern", "count"]
    counts["pattern_ru"] = counts["pattern"].map(PATTERN_RU).fillna(counts["pattern"])
    fig = px.bar(counts, x="pattern_ru", y="count", color="pattern_ru", text="count",
                 title="Распределение типов событий",
                 labels={"pattern_ru": "Тип", "count": "Кол-во"},
                 template="plotly_white",
                 color_discrete_sequence=px.colors.qualitative.Set2)
    fig.update_traces(textposition="outside")
    fig.update_layout(showlegend=False, height=400)
    return fig


def plot_f1_vs_duration(forecast_metrics, event_features):
    merged  = forecast_metrics.merge(
        event_features[["event_cluster", "total_days", "n_bursts", "pattern"]], on="event_cluster")
    pat_col = "pattern_x" if "pattern_x" in merged.columns else "pattern"
    merged["pattern_ru"] = merged[pat_col].map(PATTERN_RU).fillna(merged[pat_col])
    fig = px.scatter(
        merged, x="total_days", y="f1", color="pattern_ru", size="n_bursts",
        hover_data=["event_cluster", "mae", "rmse", "model_used"],
        title="F1 vs продолжительность события",
        labels={"total_days": "Продолжительность (дней)", "f1": "F1", "pattern_ru": "Тип"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set2, height=450,
    )
    return fig


def run_visualization(forecast_df, forecast_metrics,
                      forecast_df_prophet, forecast_metrics_prophet,
                      forecast_df_arima, forecast_metrics_arima,
                      daily, event_features, events_named,
                      output_html=None):
    if output_html is None:
        os.makedirs("reports", exist_ok=True)
        ts = datetime.now().strftime("%d-%m-%Y_%H-%M")
        output_html = f"reports/report-{ts}.html"

    name_map = dict(zip(events_named["event_cluster"], events_named["cluster_name"]))

    model_forecasts = {
        "Prophet":        forecast_df_prophet,
        "Exp. Smoothing": forecast_df,
        "ARIMA":          forecast_df_arima,
    }

    # Берём multi_burst кластеры — там разница между моделями максимальна
    mb_clusters = (
        forecast_metrics[forecast_metrics["model_used"] == "exp_smoothing"]
        .nlargest(3, "f1")["event_cluster"].tolist()
    )
    if not mb_clusters:
        mb_clusters = forecast_metrics.nlargest(3, "f1")["event_cluster"].tolist()

    print("Building figures...")
    figs, labels = [], []

    for cluster_id in mb_clusters:
        name = name_map.get(cluster_id, f"Событие #{cluster_id}")
        figs.append(plot_event_forecast_multi(
            cluster_id, daily, event_features, name_map, model_forecasts))
        labels.append(f"Прогноз: {name}")

    figs.append(plot_model_comparison({
        "Exp. Smoothing / Pattern": forecast_metrics,
        "Prophet":                  forecast_metrics_prophet,
        "ARIMA":                    forecast_metrics_arima,
    }))
    labels.append("Сравнение моделей по паттернам")

    figs.append(plot_top_events_timeline(daily, event_features, name_map))
    labels.append("Динамика топ-8 событий")

    figs.append(plot_metrics_distribution(forecast_metrics))
    labels.append("Метрики (pattern-based)")

    figs.append(plot_pattern_distribution(event_features))
    labels.append("Типы событий")

    figs.append(plot_f1_vs_duration(forecast_metrics, event_features))
    labels.append("F1 vs продолжительность")

    print(f"Saving to {output_html}...")
    with open(output_html, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'>"
                "<title>Анализ информационных событий</title></head><body>")
        f.write("<h1 style='font-family:sans-serif;padding:20px'>"
                "Моделирование информационных событий</h1>")
        for label, fig in zip(labels, figs):
            f.write(f"<h2 style='font-family:sans-serif;padding:20px 20px 0'>{label}</h2>")
            f.write(fig.to_html(
                full_html=False,
                include_plotlyjs="cdn" if label == labels[0] else False,
            ))
        f.write("</body></html>")

    print(f"Done. Open {output_html} in browser.")
    return figs


figs = run_visualization(
    forecast_df, forecast_metrics,
    forecast_df_prophet, forecast_metrics_prophet,
    forecast_df_arima, forecast_metrics_arima,
    daily, event_features,
    events_named=events_named,
)